# Multi-Agent Formations

Comprehensive examples of different agent types and formations currently supported in the agent framework. Each cell demonstrates a distinct pattern and renders the agent flow as an interactive HTML graph.

In [11]:
# Setup: Import dependencies and create mock client
from agent_framework.agents import ReActAgent, OrchestratorAgent, SequentialFlow, ParallelFlow, ConditionalFlow
from agent_framework.context import SlidingWindowContext, TokenBudgetContext, UnboundedContext, HybridContext
from agent_framework.model_clients.base_client import BaseModelClient
from agent_framework.agents.graph import FlowGraph, FlowNode, FlowEdge
from unittest.mock import MagicMock

# Mock model client (normally OpenAI, Anthropic, etc.)
client = MagicMock(spec=BaseModelClient)
print("✓ Setup complete. Ready to demonstrate agent formations.")

✓ Setup complete. Ready to demonstrate agent formations.


## 1. Single ReActAgent with SlidingWindowContext

A standalone agent using the **ReAct** (Reasoning + Acting) loop, with a sliding window memory that keeps only the last N messages.

In [2]:
# Single ReActAgent with SlidingWindowContext
agent1 = ReActAgent(
    name="researcher",
    description="Researches and gathers information",
    model_client=client,
    model_context=SlidingWindowContext(max_messages=20)
)

# Create a simple graph showing this agent
g1 = FlowGraph(name="Single ReActAgent (SlidingWindow)")
g1.add_node(FlowNode(id="input",  label="Query", node_type="input"))
g1.add_node(FlowNode(id="agent",  label="Researcher\n(ReAct)", node_type="agent"))
g1.add_node(FlowNode(id="output", label="Result", node_type="output"))
g1.add_edge(FlowEdge(source="input", target="agent"))
g1.add_edge(FlowEdge(source="agent", target="output"))

print(f"Agent: {agent1.name}")
print(f"Memory context: {agent1.model_context}")
print(f"Max messages retained: {agent1.model_context.max_messages}")
g1.draw(output='html')

Agent: researcher
Memory context: <SlidingWindowContext(max_messages=20)>
Max messages retained: 20


## 2. Single ReActAgent with TokenBudgetContext

A ReActAgent that limits memory consumption by **token count** rather than message count. Useful when working within API rate limits or cost constraints.

In [4]:
# Single ReActAgent with TokenBudgetContext
agent2 = ReActAgent(
    name="summarizer",
    description="Summarizes long documents efficiently",
    model_client=client,
    model_context=TokenBudgetContext(max_tokens=8000)
)

# Graph visualization
g2 = FlowGraph(name="Single ReActAgent (TokenBudget)")
g2.add_node(FlowNode(id="input",  label="Document", node_type="input"))
g2.add_node(FlowNode(id="agent",  label="Summarizer\n(ReAct)", node_type="agent"))
g2.add_node(FlowNode(id="output", label="Summary", node_type="output"))
g2.add_edge(FlowEdge(source="input", target="agent"))
g2.add_edge(FlowEdge(source="agent", target="output"))

print(f"Agent: {agent2.name}")
print(f"Memory context: {agent2.model_context}")
print(f"Max tokens available: {agent2.model_context.max_tokens}")
g2.draw(output='html')

Agent: summarizer
Memory context: <TokenBudgetContext(max_tokens=8000)>
Max tokens available: 8000


## 3. OrchestratorAgent with Sub-Agents

A hub-and-spoke pattern where an orchestrator agent routes tasks to multiple specialized sub-agents. The orchestrator has **UnboundedContext** by default (can be overridden).

In [6]:
# OrchestratorAgent with Sub-Agents (hub-and-spoke)
writer = ReActAgent(
    name="writer",
    description="Writes content",
    model_client=client,
    model_context=SlidingWindowContext(max_messages=10)
)

editor = ReActAgent(
    name="editor",
    description="Edits and proofreads",
    model_client=client,
    model_context=SlidingWindowContext(max_messages=10)
)

publisher = ReActAgent(
    name="publisher",
    description="Publishes content",
    model_client=client,
    model_context=SlidingWindowContext(max_messages=5)
)

orchestrator = OrchestratorAgent(
    name="content_manager",
    description="Orchestrates content creation workflow",
    model_client=client,
    sub_agents=[writer, editor, publisher]
    # orchestrator uses UnboundedContext by default
)

# Graph visualization
g3 = FlowGraph(name="OrchestratorAgent (Hub-and-Spoke)")
g3.add_node(FlowNode(id="input",       label="Request",        node_type="input"))
g3.add_node(FlowNode(id="orchestrator", label="Content\nManager", node_type="agent"))
g3.add_node(FlowNode(id="writer",       label="Writer",         node_type="agent"))
g3.add_node(FlowNode(id="editor",       label="Editor",         node_type="agent"))
g3.add_node(FlowNode(id="publisher",    label="Publisher",      node_type="agent"))
g3.add_node(FlowNode(id="output",       label="Published",      node_type="output"))

g3.add_edge(FlowEdge(source="input",       target="orchestrator"))
g3.add_edge(FlowEdge(source="orchestrator", target="writer",    label="write"))
g3.add_edge(FlowEdge(source="orchestrator", target="editor",    label="edit"))
g3.add_edge(FlowEdge(source="orchestrator", target="publisher", label="publish"))
g3.add_edge(FlowEdge(source="writer",      target="orchestrator"))
g3.add_edge(FlowEdge(source="editor",      target="orchestrator"))
g3.add_edge(FlowEdge(source="publisher",   target="output"))

print(f"Orchestrator: {orchestrator.name}")
print(f"Sub-agents: {[a.name for a in orchestrator.sub_agents]}")
print(f"Orchestrator context: {orchestrator.model_context}")
g3.draw(output='html')

Orchestrator: content_manager
Sub-agents: ['writer', 'editor', 'publisher']
Orchestrator context: <UnboundedContext>


## 4. SequentialFlow: Linear Agent Pipeline

Agents execute one after another in strict order. Each agent's output becomes the input to the next. Perfect for assembly-line workflows.

In [7]:
# SequentialFlow: Linear pipeline (Agent A → Agent B → Agent C)
analyst = ReActAgent(
    name="analyst",
    description="Analyzes data",
    model_client=client,
    model_context=SlidingWindowContext(max_messages=15)
)

visualizer = ReActAgent(
    name="visualizer",
    description="Creates visualizations",
    model_client=client,
    model_context=SlidingWindowContext(max_messages=15)
)

formatter = ReActAgent(
    name="formatter",
    description="Formats final output",
    model_client=client,
    model_context=SlidingWindowContext(max_messages=15)
)

sequential_flow = SequentialFlow(
    steps=[analyst, visualizer, formatter],
    name="data_pipeline",
    description="Data analysis pipeline"
)

# Graph visualization
g4 = FlowGraph(name="SequentialFlow (Linear Pipeline)")
g4.add_node(FlowNode(id="input",      label="Raw Data",      node_type="input"))
g4.add_node(FlowNode(id="analyst",    label="Analyst",       node_type="agent"))
g4.add_node(FlowNode(id="visualizer", label="Visualizer",    node_type="agent"))
g4.add_node(FlowNode(id="formatter",  label="Formatter",     node_type="agent"))
g4.add_node(FlowNode(id="output",     label="Report",        node_type="output"))

g4.add_edge(FlowEdge(source="input",      target="analyst"))
g4.add_edge(FlowEdge(source="analyst",    target="visualizer"))
g4.add_edge(FlowEdge(source="visualizer", target="formatter"))
g4.add_edge(FlowEdge(source="formatter",  target="output"))

print(f"Sequential Flow: {sequential_flow.name}")
print(f"Steps (in order): {[a.name for a in sequential_flow.steps]}")
g4.draw(output='html')

Sequential Flow: data_pipeline
Steps (in order): ['analyst', 'visualizer', 'formatter']


## 5. ParallelFlow: Concurrent Execution

Multiple agents process tasks **simultaneously** and their results are merged. Use when tasks are independent and can run concurrently for speed.

In [8]:
# ConditionalFlow: Dynamic routing based on conditions (predicate → if_true or if_false)
question_answerer = ReActAgent(
    name="qa_agent",
    description="Answers factual questions",
    model_client=client,
    model_context=SlidingWindowContext(max_messages=10)
)

creative_writer = ReActAgent(
    name="creative_agent",
    description="Generates creative content",
    model_client=client,
    model_context=SlidingWindowContext(max_messages=15)
)

code_generator = ReActAgent(
    name="code_gen_agent",
    description="Generates code",
    model_client=client,
    model_context=SlidingWindowContext(max_messages=20)
)

# Define a routing predicate — check if input is a code-related question
def is_code_request(text: str) -> bool:
    keywords = ["code", "python", "function", "debug", "algorithm", "class"]
    return any(kw.lower() in text.lower() for kw in keywords)

# Create conditional flow: if code → code_generator, else → use a parallel of QA + creative
qa_or_creative = ParallelFlow(
    branches=[question_answerer, creative_writer],
    name="qa_or_creative",
    merge="concat"
)

conditional_flow = ConditionalFlow(
    predicate=is_code_request,
    if_true=code_generator,
    if_false=qa_or_creative,
    name="smart_router",
    description="Routes based on input type"
)

# Graph visualization (simplified view of conditional routing)
g6 = FlowGraph(name="ConditionalFlow (Decision Routing)")
g6.add_node(FlowNode(id="input",       label="User Request", node_type="input"))
g6.add_node(FlowNode(id="router",      label="Is Code\nRequest?", node_type="condition"))
g6.add_node(FlowNode(id="code_gen",    label="Code Gen\nAgent",     node_type="agent"))
g6.add_node(FlowNode(id="qa",          label="Q&A Agent",    node_type="agent"))
g6.add_node(FlowNode(id="creative",    label="Creative\nAgent",     node_type="agent"))
g6.add_node(FlowNode(id="output",      label="Response",    node_type="output"))

# Router decides which path to take
g6.add_edge(FlowEdge(source="input",   target="router"))
g6.add_edge(FlowEdge(source="router",  target="code_gen", label="yes"))
g6.add_edge(FlowEdge(source="router",  target="qa",       label="no"))
g6.add_edge(FlowEdge(source="router",  target="creative", label="no"))

# All paths converge at output
g6.add_edge(FlowEdge(source="code_gen", target="output"))
g6.add_edge(FlowEdge(source="qa",       target="output"))
g6.add_edge(FlowEdge(source="creative", target="output"))

print(f"Conditional Flow: {conditional_flow.name}")
print(f"Predicate: is_code_request(text)")
print(f"True branch: {conditional_flow.if_true.name}")
print(f"False branch: {conditional_flow.if_false.name}")
g6.draw(output='html')

Conditional Flow: smart_router
Predicate: is_code_request(text)
True branch: code_gen_agent
False branch: qa_or_creative


## 6. ConditionalFlow: Decision-Driven Routing

Agents are selected based on **conditions evaluated at runtime**. Routes input to different agents depending on the input properties or decision logic.

In [9]:
# ParallelFlow: Concurrent execution (Agent A || Agent B || Agent C)
code_reviewer = ReActAgent(
    name="code_reviewer",
    description="Reviews code quality",
    model_client=client,
    model_context=TokenBudgetContext(max_tokens=5000)
)

security_auditor = ReActAgent(
    name="security_auditor",
    description="Audits security",
    model_client=client,
    model_context=TokenBudgetContext(max_tokens=5000)
)

performance_analyzer = ReActAgent(
    name="performance_analyzer",
    description="Analyzes performance",
    model_client=client,
    model_context=TokenBudgetContext(max_tokens=5000)
)

parallel_flow = ParallelFlow(
    branches=[code_reviewer, security_auditor, performance_analyzer],
    name="code_review_pipeline",
    description="Parallel code review pipeline",
    merge="concat"
)

# Graph visualization
g5 = FlowGraph(name="ParallelFlow (Concurrent Execution)")
g5.add_node(FlowNode(id="input",              label="Code",                node_type="input"))
g5.add_node(FlowNode(id="code_reviewer",     label="Code\nReviewer",       node_type="agent"))
g5.add_node(FlowNode(id="security_auditor",  label="Security\nAuditor",    node_type="agent"))
g5.add_node(FlowNode(id="perf_analyzer",     label="Performance\nAnalyzer", node_type="agent"))
g5.add_node(FlowNode(id="output",            label="Report",              node_type="output"))

# All agents process input in parallel
g5.add_edge(FlowEdge(source="input", target="code_reviewer"))
g5.add_edge(FlowEdge(source="input", target="security_auditor"))
g5.add_edge(FlowEdge(source="input", target="perf_analyzer"))

# All results merge at output
g5.add_edge(FlowEdge(source="code_reviewer",    target="output"))
g5.add_edge(FlowEdge(source="security_auditor", target="output"))
g5.add_edge(FlowEdge(source="perf_analyzer",    target="output"))

print(f"Parallel Flow: {parallel_flow.name}")
print(f"Concurrent branches: {[a.name for a in parallel_flow.branches]}")
print(f"Merge strategy: {parallel_flow.merge}")
g5.draw(output='html')

Parallel Flow: code_review_pipeline
Concurrent branches: ['code_reviewer', 'security_auditor', 'performance_analyzer']
Merge strategy: concat


## 7. Complex Hierarchical Formation

Combining multiple patterns: An OrchestratorAgent that manages both sequential and parallel sub-flows. This demonstrates real-world complexity.

In [10]:
# Complex hierarchical: OrchestratorAgent managing multiple flows
# Build a sequential data pipeline
etl_extract = ReActAgent(
    name="extract",
    description="Extracts data from sources",
    model_client=client,
    model_context=SlidingWindowContext(max_messages=10)
)

etl_transform = ReActAgent(
    name="transform",
    description="Transforms data",
    model_client=client,
    model_context=SlidingWindowContext(max_messages=10)
)

# Parallel QA agents (independent)
qa1 = ReActAgent(
    name="qa_accuracy",
    description="Tests accuracy",
    model_client=client,
    model_context=SlidingWindowContext(max_messages=8)
)

qa2 = ReActAgent(
    name="qa_completeness",
    description="Tests completeness",
    model_client=client,
    model_context=SlidingWindowContext(max_messages=8)
)

# Final export
exporter = ReActAgent(
    name="exporter",
    description="Exports results",
    model_client=client,
    model_context=SlidingWindowContext(max_messages=5)
)

# All agents managed by orchestrator
master_orchestrator = OrchestratorAgent(
    name="master_pipeline",
    description="Manages ETL + QA + Export",
    model_client=client,
    sub_agents=[etl_extract, etl_transform, qa1, qa2, exporter]
)

# Graph visualization of the complex hierarchy
g7 = FlowGraph(name="Complex Hierarchical Formation")
g7.add_node(FlowNode(id="input",        label="Source Data", node_type="input"))
g7.add_node(FlowNode(id="master",       label="Master\nOrchestrator", node_type="agent"))

# Sequential ETL pipeline
g7.add_node(FlowNode(id="extract",      label="Extract",     node_type="agent"))
g7.add_node(FlowNode(id="transform",    label="Transform",   node_type="agent"))

# Parallel QA agents
g7.add_node(FlowNode(id="qa_acc",       label="QA:\nAccuracy", node_type="agent"))
g7.add_node(FlowNode(id="qa_comp",      label="QA:\nCompleteness", node_type="agent"))

# Export
g7.add_node(FlowNode(id="export",       label="Export",      node_type="agent"))
g7.add_node(FlowNode(id="output",       label="Final Data",  node_type="output"))

# Data flows
g7.add_edge(FlowEdge(source="input", target="master"))
g7.add_edge(FlowEdge(source="master", target="extract"))
g7.add_edge(FlowEdge(source="extract", target="transform", label="sequential"))
g7.add_edge(FlowEdge(source="transform", target="qa_acc"))
g7.add_edge(FlowEdge(source="transform", target="qa_comp", label="parallel"))
g7.add_edge(FlowEdge(source="qa_acc", target="export"))
g7.add_edge(FlowEdge(source="qa_comp", target="export"))
g7.add_edge(FlowEdge(source="export", target="output"))

print(f"Master Orchestrator: {master_orchestrator.name}")
print(f"Total agents: {len(master_orchestrator.sub_agents)}")
print(f"Pattern: Sequential ETL → Parallel QA → Export")
g7.draw(output='html')

Master Orchestrator: master_pipeline
Total agents: 5
Pattern: Sequential ETL → Parallel QA → Export


## Summary: Memory Context Options

Each agent in the formations above uses a specific **memory context** strategy:

| Context Type | Use Case | Configuration |
|---|---|---|
| **SlidingWindowContext** | Recent conversation history | `max_messages=N` — keeps last N messages |
| **TokenBudgetContext** | Cost/rate-limit control | `max_tokens=N` — limits total tokens in context |
| **UnboundedContext** | Full history (default for Orchestrator) | No limit — keeps all messages |
| **HybridContext** | Balanced approach | Combines recent + total token limits |

All agents now **require** a `model_context` parameter at initialization. No defaults for ReActAgent/BaseAgent—explicit is better than implicit!